#### **What is `dbt compile` used for?**

**Short answer (intuition first)**

> `dbt compile` **converts your dbt project (Jinja + macros + refs + sources) into pure SQL files — without running anything in the database.**

No tables created ❌

No queries executed ❌

Only SQL generation ✅



----------


**Why `dbt compile` exists (the real reason)**

dbt models are not plain SQL. They contain:

- Jinja (`{{ }}`)

- `ref()` and `source()`

- conditionals (`is_incremental()`)

- configs (`materialized`, `tags`, etc.)

Your data warehouse **cannot understand these.**

So dbt must first **compile** everything into **raw SQL** that Snowflake / BigQuery / Postgres can execute.

--------

#### **What exactly happens during `dbt compile`**

When you run:

In [ ]:
dbt compile

dbt does 4 critical things:

**1️⃣ Resolves `ref()` and `source()`**

Example model:

In [ ]:
SELECT *
FROM {{ source('airbnb', 'reviews') }}

Compiled SQL:

In [ ]:
SELECT *
FROM AIRBNB.RAW.RAW_REVIEWS

dbt figures out:

- database

- schema

- table name

- environment (dev / prod)

--------

**2️⃣ Evaluates Jinja logic**

Example:

In [ ]:
SELECT *
FROM {{ ref('stg_reviews') }}
{% if is_incremental() %}
  WHERE review_date > (SELECT MAX(review_date) FROM {{ this }})
{% endif %}

During **compile**:

- `is_incremental()` is evaluated

- dbt decides **which SQL version is valid**

- outputs final executable SQL

-------

#### **3️⃣ Applies configs**

Example:

In [ ]:
{{ config(materialized='table') }}

dbt converts this into:

- CREATE TABLE / CREATE VIEW

- schema resolution

- naming logic

All of this is reflected in compiled SQL.

-----------

#### **4️⃣ Writes output to `target/compiled/`**

After compilation, dbt saves SQL files here:

In [ ]:
target/
 └── compiled/
     └── airbnb/
         └── models/
             └── fct_reviews.sql

👉 These files are **exactly what dbt would run** if you used `dbt run`.

--------

**What `dbt compile` does NOT do (very important)**

| Action              | Happens? |
| ------------------- | -------- |
| Create tables/views | ❌ No     |
| Insert/update data  | ❌ No     |
| Run queries         | ❌ No     |
| Touch warehouse     | ❌ No     |

👉 It is **100% safe** to run anytime.

---------------

#### **When should you use dbt compile? (Real-world cases)**

**✅ Case 1: Debugging Jinja errors**

If dbt fails with:

- syntax errors

- macro issues

- ref/source resolution errors

Run:

In [ ]:
dbt compile

You’ll see **exactly which model breaks**, without executing data.

----------

**✅ Case 2: Understanding what SQL dbt generates**

This is HUGE for beginners.

You write this:

In [ ]:
SELECT *
FROM {{ ref('stg_hosts') }}

But Snowflake runs:

In [ ]:
SELECT *
FROM AIRBNB.DEV.STG_HOSTS

`dbt compile` lets you **see the truth**.

----------

**✅ Case 3: CI / code reviews**

In production teams:

- PR checks run dbt compile

- Ensures project compiles successfully

- Faster than dbt run

- No warehouse cost

-----------

**✅ Case 4: Testing environment differences**

Dev vs Prod differences in:

- schemas

- databases

- configs

Compile shows **environment-specific SQL.**

-----------

**`dbt compile` vs `dbt run`**

| Command       | What it does                     |
| ------------- | -------------------------------- |
| `dbt compile` | Generate SQL only                |
| `dbt run`     | Compile + execute SQL            |
| `dbt test`    | Compile + run tests              |
| `dbt build`   | Compile + run + test + snapshots |


**👉 All dbt commands compile first.**

`dbt compile` just stops there.